<a href="https://colab.research.google.com/github/UL0222199/Podstawy_SI/blob/main/eliza_ns_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ELIZA i LLM — od reguł do złudzenia rozumienia

**Podstawy AI · Laboratorium 1 (studia niestacjonarne)**

Dzisiejsze zajęcia mają dwie części z przerwą pośrodku:

| Część | Co robisz |
|---|---|
| **Część I — ELIZA** | |
| 1. Trochę historii | czytasz (5 min) |
| 2. Wyrażenia regularne | uruchamiasz przykłady |
| 3. Budowa ELIZY | Zadania 1–2: dopisujesz zaimki i reguły, testujesz |
| 4. Złam ELIZĘ | Zadanie 3 |
| ☕ **Przerwa** | |
| **Część II — ELIZA kontra LLM** | |
| 5. Połączenie z modelem przez API | klucz, klient, test połączenia |
| 6. Ta sama rozmowa, dwa silniki | Zadanie 4 |
| 7. Łamiące zdania — druga runda | Zadanie 5 |
| 8. Pamięć to lista wiadomości | Zadanie 6 |
| 9. Rola to tekst | Zadanie 7 |
| 10. Refleksja | Zadanie 8 |

Kod ELIZY jest **prawie gotowy** — Twoim zadaniem jest go uzupełnić, uruchomić,
przetestować i zrozumieć, *dlaczego* działa (albo nie działa).

**Potrzebujesz:** klucza Gemini API do części II (instrukcja na slajdach / mailu).

> Teza, którą sprawdzamy przez całe zajęcia: **wrażenie rozumienia powstaje
> po stronie człowieka, nie maszyny.**

# Część I — ELIZA

## 1. Trochę historii

**Kto i kiedy.** ELIZĘ napisał **Joseph Weizenbaum** w MIT w latach 1964–1966.
Opis ukazał się w styczniu 1966 r. w *Communications of the ACM*. Program działał
na komputerze IBM 7094 — użytkownik pisał na dalekopisie, a odpowiedź
drukowała się po chwili.

**Skąd nazwa.** Od Elizy Doolittle z *Pigmaliona* G. B. Shawa — kwiaciarki,
którą uczy się „mówić jak dama". Program też można było „uczyć" mówić,
podmieniając mu zestaw reguł (tzw. *skrypt*).

**DOCTOR.** Najsłynniejszy skrypt udawał psychoterapeutę w nurcie
**rogeriańskim** (Carl Rogers). Taki terapeuta niewiele mówi od siebie —
głównie odbija i parafrazuje słowa pacjenta: *„Mówisz, że…?"*,
*„Jak się z tym czujesz?"*. W tej roli milczenie o świecie wygląda na
profesjonalizm, a nie na niewiedzę. Program nie musiał nic wiedzieć —
wystarczyło przekształcać zdania.

**Efekt ELIZY.** Ludzie bardzo szybko przypisywali programowi zrozumienie
i empatię. Sekretarka Weizenbauma — która widziała, jak program powstaje —
po kilku wymianach zdań poprosiła go, by wyszedł z pokoju. Tę skłonność
nazywamy dziś **efektem ELIZY**.

**Reakcja twórcy.** Weizenbauma to zaniepokoiło. W książce
*Computer Power and Human Reason* (1976) krytykował pomysł, by maszynom
powierzać decyzje wymagające ludzkiego osądu i troski.

**Jak działa** — trzy operacje, nic więcej:

1. **słowo kluczowe** — znajdź w zdaniu wyzwalacz,
2. **dekompozycja** — dopasuj wzorzec z „dziurą", np. `potrzebuję (.*)`,
3. **rekompozycja** — wstaw wycięty fragment do szablonu, odwracając zaimki.

Nie ma tu pamięci, uczenia ani reprezentacji znaczenia.

## 2. Wyrażenia regularne — tyle, ile potrzeba

**Wyrażenie regularne** (*regex*) to wzorzec opisujący, jak wygląda fragment
tekstu. W Pythonie obsługuje je moduł `re`. Wzorce zapisujemy jako surowe
napisy `r"..."`.

| Wzorzec | Znaczenie | Przykład |
|---|---|---|
| `.` | dowolny jeden znak | `k.t` → kot, kit |
| `*` | poprzedni element 0 lub więcej razy | `ha*` → h, ha, haaa |
| `.*` | „cokolwiek" — nasza **dziura** | |
| `( )` | **grupa** — zapamiętaj, co pasowało | `potrzebuję (.*)` |
| `a\|b` | a **albo** b | `matka\|ojciec` |
| `\b` | granica słowa | `\bkot\b` nie trafi w „kotlet" |
| `[^\w\s]` | znak, który nie jest literą ani spacją (interpunkcja) | |

| Funkcja | Co robi |
|---|---|
| `re.search(wzorzec, tekst)` | szuka wzorca gdziekolwiek w tekście; zwraca dopasowanie albo `None` |
| `m.group(1)` / `m.groups()` | fragment złapany przez grupę / krotka wszystkich grup |
| `re.sub(wzorzec, zamiennik, tekst)` | zamienia wystąpienia wzorca |

Uruchom komórki i przyjrzyj się wynikom.

In [ ]:
import re
import random

m = re.search(r"potrzebuję (.*)", "chyba potrzebuję urlopu")
print(m.group(0))   # całe dopasowanie
print(m.group(1))   # to, co złapała grupa (.*)
print(m.groups())   # krotka wszystkich grup

print(re.search(r"potrzebuję (.*)", "wszystko w porządku"))   # brak dopasowania -> None

potrzebuję urlopu
urlopu
('urlopu',)
None


In [ ]:
# Dwie pułapki: przecinek po słowie kluczowym i kropka na końcu
print(re.search(r"potrzebuję (.*)", "potrzebuję, żeby ktoś mnie słuchał"))   # None!
print(re.search(r"czuję się (.*)", "czuję się samotny.").group(1))            # z kropką

# Rozwiązanie: NORMALIZUJEMY tekst — małe litery, bez interpunkcji
def normalizuj(tekst):
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

print(normalizuj("Potrzebuję, żeby ktoś mnie słuchał!"))

None
samotny.
potrzebuję żeby ktoś mnie słuchał


In [ ]:
# Wypełnianie szablonu: .format(*grupy) wstawia grupy w miejsca {0}, {1}, ...
szablon = "Dlaczego chciałbyś {0}?"
grupy = ["mieć więcej czasu"]
print(szablon.format(*grupy))   # poprawnie
print(szablon.format(grupy))    # bez * — cała lista trafia do {0}

Dlaczego chciałbyś mieć więcej czasu?
Dlaczego chciałbyś ['mieć więcej czasu']?


## 3. Budowa ELIZY

Poniżej jest **cała ELIZA**. Brakuje tylko „wiedzy": słownika zaimków
i zestawu reguł. Najpierw przeczytaj kod funkcji i upewnij się, że rozumiesz
każdą linijkę — o to mogę zapytać na zajęciach.

### Mechanizm 1 — odbicie zaimków

ELIZA odwraca perspektywę: *„moja rodzina mnie słucha"* → *„twoja rodzina
ciebie słucha"*. Działa to **w obie strony**: gdy pacjent mówi o terapeucie
(*„o co **ci** chodzi"*), terapeuta odpowiada o sobie (*„o co **mi** chodzi"*).

Funkcja `odbij` jest gotowa. `odbicia.get(s, s)` zwraca odbicie słowa,
a jeśli słowa nie ma w słowniku — samo słowo bez zmian.

### 🔧 Zadanie 1
Uzupełnij słownik `odbicia` — dodaj **co najmniej 6** par, w tym
**co najmniej 2 w kierunku odwrotnym** (od terapeuty do pacjenta, np. `ci → mi`).
Uruchom test pod spodem.

In [ ]:
odbicia = {
    # pacjent -> terapeuta
    "jestem": "jesteś",
    "moja": "twoja",
    "mnie": "ciebie",
    # TODO (Zadanie 1): dopisz min. 6 par, np. mój/twój, moje/twoje, mam/masz, mi/ci, ja/ty
    "mi": "ci",
    "ja": "ty",
    "nasze": "wasze",
    "wiem": "wiesz",
    "czuję": "czujesz",
    "myślę": "myślisz",
    "moje": "twoje",
    "mam": "masz",
    "doświadczyłem": "doświadczyłeś",
    # oraz min. 2 w kierunku odwrotnym, np. "ci": "mi", "tobie": "mnie"
    "ci": "mi",
    "tobie": "mnie",
    "słuchasz": "słucham",
    "dasz": "dam"
}

def odbij(fragment):
    slowa = fragment.lower().split()
    return " ".join(odbicia.get(s, s) for s in slowa)

In [ ]:
# Test Zadania 1 — powinno przejść bez błędu:
assert odbij("moja rodzina") == "twoja rodzina"
assert odbij("o co ci chodzi") == "o co mi chodzi"   # kierunek odwrotny!
print(odbij("jestem zmęczony i moja praca mnie męczy"))
print("Zadanie 1: OK")

jesteś zmęczony i twoja praca ciebie męczy
Zadanie 1: OK


### Mechanizmy 2 + 3 — reguły i odpowiedź

Każda reguła to para: **wzorzec** (regex) + **lista szablonów** odpowiedzi.
Funkcja `odpowiedz` (gotowa) sprawdza reguły **po kolei** — wygrywa
**pierwsza** pasująca. Z niej bierze grupy, odbija w nich zaimki i wstawia
do losowo wybranego szablonu.

Ostatnia reguła `r".*"` to **reguła awaryjna** — pasuje do każdego tekstu,
dzięki czemu ELIZA zawsze coś odpowiada.

### 🔧 Zadanie 2
Dopisz **co najmniej 4** własne reguły **przed** regułą awaryjną.
- tekst jest znormalizowany, więc wzorce piszemy **małymi literami, bez interpunkcji**,
- sprawdź, czy szablony są gramatyczne po wstawieniu fragmentu,
- odpowiedz w komentarzu: **dlaczego reguła awaryjna musi być na końcu?**

In [ ]:
# (wzorzec, [szablony]) — dopasowujemy na tekście po normalizuj()
reguly = [
    (r"potrzebuję (.*)",
        ["Dlaczego potrzebujesz {0}?", "Czy naprawdę pomogłoby ci {0}?"]),
    (r"czuję się (.*)",
        ["Od jak dawna czujesz się {0}?", "Czy często tak się czujesz?"]),
    (r"\b(matka|ojciec|dziadek|rodzina|siostra|brat)\b",
      ["Opowiedz mi więcej o swojej rodzinie.", "Jak układają się te relacje?"]),
    # TODO (Zadanie 2): dodaj min. 4 własne reguły, np. „boję się (.*)", „chciałbym (.*)",
    #                   „jestem (.*)", „nie wiem", „wszyscy|nikt|zawsze|nigdy" ...
    (r"mam marzenie aby (.*)", ["Skąd wzięło się to pragnienie by {0}?", "Rozumiem, że jest to dla Ciebie bardzo ważnym, aby {0}. Opowiedz mi o tym więcej."]),
    (r"mam problem z (.*)", ["Możesz opowiedzieć mi więcej o tym problemie?", "Co rozumiesz poprzez {0}?"]),
    (r"nie wiem", ["To zrozumiałe. Co mogłoby pomóc ci się dowiedzieć?"]),
    (r"jestem (.*)", ["Dlaczego uważasz, że jesteś {0}?"]),
    (r"\b(szef|współpracownik)\b", ["Opowiedz mi więcej o swojej pracy"]),
    (r"ciągle|zawsze", ["Kiedy ostatnio się to zdarzyło?", "Jak się z tym czujesz?"]),
    (r"boję się (.*)", ["Czy uważasz, że to {0} naprawdę mogłoby się wydarzyć?"]),
    (r".*",   # reguła awaryjna — ZAWSZE na końcu
        ["Rozumiem. Mów dalej.", "Co masz na myśli?", "Jak się z tym czujesz?"]),
]
# Dlaczego reguła awaryjna musi być na końcu?
# Ponieważ funkcja `odpowiedz` sprawdza reguły po kolei i wybiera pierwszą pasującą.
# Wzorzec reguły awaryjnej pasuje do każdego tekstu i gdyby był wyżej, każda reguła poniżej byłaby nieosiągalna.

def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            grupy = [odbij(g) for g in m.groups()]
            return random.choice(szablony).format(*grupy)

In [ ]:
TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]

for zdanie in TESTY:
    print(f"Pacjent: {zdanie}\nELIZA:   {odpowiedz(zdanie)}\n")

Pacjent: Potrzebuję, żeby moja rodzina mnie słuchała.
ELIZA:   Czy naprawdę pomogłoby ci żeby twoja rodzina ciebie słuchała?

Pacjent: Czuję się dzisiaj samotny.
ELIZA:   Od jak dawna czujesz się dzisiaj samotny?

Pacjent: Mój ojciec ciągle mnie krytykuje.
ELIZA:   Jak układają się te relacje?

Pacjent: Nie wiem, o co ci chodzi.
ELIZA:   To zrozumiałe. Co mogłoby pomóc ci się dowiedzieć?

Pacjent: Wszyscy mnie ignorują.
ELIZA:   Jak się z tym czujesz?

Pacjent: Boję się, że mnie zostawią.
ELIZA:   Czy uważasz, że to że ciebie zostawią naprawdę mogłoby się wydarzyć?



In [ ]:
# (Opcjonalnie) Rozmowa na żywo. Wpisz 'koniec', by wyjść.
# UWAGA: przy „Run All" ta komórka zatrzyma notebook — uruchamiaj ją ręcznie.
while True:
    tekst = input("Ty: ")
    if tekst.strip().lower() == "koniec":
        break
    print("ELIZA:", odpowiedz(tekst))

## 4. Złam ELIZĘ

Teraz pracujesz jak adwersarz: **celowo** szukasz wypowiedzi, które obnażają
brak rozumienia. Te zdania przydadzą się po przerwie — puścimy je przez LLM.

### 🔧 Zadanie 3 (kod + odpowiedź pisemna)
Znajdź **3–4 różne typy** sytuacji, w których iluzja pęka. Dla każdej podaj
zdanie, pokaż odpowiedź ELIZY i wyjaśnij, **dlaczego** mechanizm tu zawodzi
(odwołaj się do konkretnej linijki kodu).

Gdzie szukać: negacja · odwołanie do czegoś sprzed kilku zdań (brak pamięci) ·
ironia, żart · niegramatyczne odbicie zaimków · fragment niepasujący do szablonu.

In [ ]:
przyklady_lamiace = [
    # TODO (Zadanie 3): wstaw tu swoje zdania łamiące
   "Czuje się źle sam ze sobą",
   "Mój dziadek niedawno umarł, mogłabyś pomóc mi to przetrawić?",
   "Co masz na myśli?"
]
for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n")

> Czuje się źle sam ze sobą
  ELIZA: Jak się z tym czujesz?

> Mój dziadek niedawno umarł, mogłabyś pomóc mi to przetrawić?
  ELIZA: Jak układają się te relacje?

> Co masz na myśli?
  ELIZA: Rozumiem. Mów dalej.



**Twoja analiza (Zadanie 3):**

1. Typ: literówka w zdaniu
— przykład: "Czuje się źle sam ze sobą" -> "Jak się z tym czujesz?"
— dlaczego zawodzi: Jest zdefiniowana reguła dla słowa `czuję`, ale nie `czuje`.
2. Typ: brak rozumienia kontekstu
— przykład: "Mój dziadek niedawno umarł, mogłabyś pomóc mi to przetrawić?" -> "Jak układają się te relacje?"
— dlaczego zawodzi: Reguły są sztywne, wyłapują wyrażenie, a następnie Eliza odpowiada w ściścle określony sposób, nie potrafi się dostosowywać do konkretnej wypowiedzi.
3. Typ: brak pamięci
— przykład: "Co masz na myśli?" -> "Jak się z tym czujesz?"
— dlaczego zawodzi: Eliza nie pamięta, ani nie rozumie rozmowy.

---

# ☕ Przerwa

Przed powrotem przygotuj **klucz Gemini API** (aistudio.google.com → *Get API key*).

---

# Część II — ELIZA kontra LLM

Te same zdania trafią teraz do dużego modelu językowego (LLM).
Pytanie na tę część: **które pęknięcia ELIZY LLM naprawia, a które zostają?**

> ⚠️ **Nie wpisuj prawdziwych, osobistych zwierzeń.** W darmowym planie
> dostawca może wykorzystywać przesłane treści do ulepszania usług.
> Rozmawiamy z „terapeutą", ale wszystkie zdania mają być **wymyślone**.

## 5. Połączenie z modelem przez API

**Czym jest LLM.** Duży model językowy to sieć neuronowa wytrenowana na
ogromnych zbiorach tekstu do przewidywania, jaki fragment tekstu (*token*)
powinien pojawić się dalej. Odpowiedź powstaje token po tokenie, a wybór
jest częściowo losowy. Nie ma tu reguł pisanych ręcznie jak w ELIZIE.

**Zapytanie do API** to **lista wiadomości** z rolami:

| Rola | Kto mówi |
|---|---|
| `system` | instrukcja dla modelu — kim ma być, jak odpowiadać |
| `user` | użytkownik |
| `assistant` | wcześniejsze odpowiedzi modelu |

W odpowiedzi dostajemy **jedną** nową wiadomość modelu. Zapamiętaj ten
schemat — w Zadaniu 6 okaże się ważniejszy, niż wygląda.

**Klucz** podajesz przez `getpass` albo Colab *Secrets* (ikona klucza po lewej,
nazwa `GEMINI_API_KEY`). **Nigdy nie wklejaj klucza wprost do kodu.**

In [ ]:
# !pip install openai        # odkomentuj, jeśli biblioteki nie ma (w Colabie jest)
import os
import time
from getpass import getpass
from openai import OpenAI

try:                                    # Colab: klucz z Secrets
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:                       # lokalnie: zmienna środowiskowa albo getpass
    API_KEY = os.environ.get("GEMINI_API_KEY") or getpass("Wklej klucz Gemini API: ")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.5-flash-lite"   # jeśli nie działa: sprawdź listę w następnej komórce
REASONING = "minimal"             # ogranicza „myślenie" modelu; przy błędzie 400 ustaw None
print("Klient gotowy. Model:", MODEL)

Klient gotowy. Model: gemini-3.5-flash-lite


In [ ]:
# Jakie modele Flash są dostępne dla Twojego klucza?
for m in client.models.list():
    if "flash" in m.id:
        print(m.id)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [ ]:
def czat(messages, temperatura=None, max_tokens=400):
    params = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
    if REASONING:
        params["reasoning_effort"] = REASONING
    if temperatura is not None:
        params["temperature"] = temperatura
    for proba in range(3):
        try:
            r = client.chat.completions.create(**params)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "429" in str(e) and proba < 2:
                print("  (limit zapytań — czekam 20 s)")
                time.sleep(20)
            else:
                raise

SYSTEM_TERAPEUTA = ("Jesteś terapeutą prowadzącym rozmowę po polsku. "
                    "Odpowiadaj krótko, jednym–dwoma zdaniami.")

def zapytaj_llm(tekst, system=SYSTEM_TERAPEUTA, temperatura=2.4):
    # JEDNO pytanie, BEZ historii rozmowy
    return czat([{"role": "system", "content": system},
                 {"role": "user",   "content": tekst}])

print(zapytaj_llm("Czuję się dzisiaj samotny."))   # test połączenia

Przykro mi, że tak się czujesz. Chcesz o tym opowiedzieć?


**Coś nie działa?**

| Objaw | Przyczyna |
|---|---|
| `401` / `API key not valid` | źle skopiowany klucz — uruchom komórkę z kluczem jeszcze raz |
| `404` / `model not found` | nazwa modelu się zmieniła — wybierz z listy wyżej |
| `400` z `reasoning_effort` | ustaw `REASONING = None` i uruchom komórki ponownie |
| `429` | limit zapytań — odczekaj minutę; nie uruchamiaj pętli wielokrotnie |
| pusta odpowiedź `""` | model zużył limit tokenów na „myślenie" — zwiększ `max_tokens` |

## 6. Ta sama rozmowa, dwa silniki

### 🔧 Zadanie 4
1. Uzupełnij pętlę tak, żeby każde zdanie z `TESTY` trafiło do **obu** systemów.
2. Zapisz **3 konkretne różnice**. Nie oceniaj, który jest „lepszy" —
   opisz, **co** każdy z nich robi z Twoim zdaniem.

*Na co patrzeć:* czy odpowiedź powtarza Twoje słowa, czy dodaje coś nowego?
Czy jest gramatyczna? Czy odnosi się do sensu, czy do słowa kluczowego?

In [ ]:
for zdanie in TESTY:
    odp_eliza = odpowiedz(zdanie)
    odp_llm = zapytaj_llm(zdanie)         # <- uzupełnij (Zadanie 4)
    print(f"> {zdanie}\n  ELIZA: {odp_eliza}\n  LLM:   {odp_llm}\n")

> Potrzebuję, żeby moja rodzina mnie słuchała.
  ELIZA: Dlaczego potrzebujesz żeby twoja rodzina ciebie słuchała?
  LLM:   To naturalna potrzeba, że chcesz być usłyszany i zrozumiany przez najbliższych. Co dokładnie czujesz, kiedy masz wrażenie, że Twoje zdanie nie dociera do nich?

> Czuję się dzisiaj samotny.
  ELIZA: Od jak dawna czujesz się dzisiaj samotny?
  LLM:   Rozumiem, to trudne uczucie. Chcesz o tym opowiedzieć?

> Mój ojciec ciągle mnie krytykuje.
  ELIZA: Opowiedz mi więcej o swojej rodzinie.
  LLM:   To musi być bardzo bolesne, kiedy bliska osoba zamiast wsparcia daje ci poczucie ciągłej oceny. Co dokładnie zazwyczaj wzbudza w nim najwięcej krytyki?

> Nie wiem, o co ci chodzi.
  ELIZA: To zrozumiałe. Co mogłoby pomóc ci się dowiedzieć?
  LLM:   Rozumiem, że możesz czuć się zagubiony w tym momencie. O czym chciałbyś teraz porozmawiać?

> Wszyscy mnie ignorują.
  ELIZA: Jak się z tym czujesz?
  LLM:   To musi być bardzo bolesne uczucie poczucia nieważności. Chcesz mi o ty

**Różnice (Zadanie 4):**

1. Eliza powtarza słowa, nie dodaje nic nowego. LLM potrafi zapytać/dopowiedzieć coś co nawiązuje do pytania, ale nie jest bezpośrednio w nim.
2. LLM zawsze na początku parafrazuje/podsumowuje wypowiedź pacjenta, co daje poczucie głębszego zrozumienia w porównaniu do Elizy, która najczęściej odpowiada jednym zdaniem i nie wnosi niczego nowego.
3. Elizie zdarzają się błędny składniowe/gramatyczne, LLM odpowiada gramatycznie bezbłędnie.

## 7. Łamiące zdania — druga runda

### 🔧 Zadanie 5
1. Uruchom porównanie na **Twoich** zdaniach z Zadania 3 (lista `przyklady_lamiace`
   jest już zdefiniowana wyżej).
2. **Zanim uruchomisz** test pamięci, zapisz swoje przewidywanie:
   czy LLM odpowie poprawnie na drugie pytanie?
3. Uzupełnij tabelę.

In [ ]:
for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n  LLM:   {zapytaj_llm(zdanie)}\n")

> Czuje się źle sam ze sobą
  ELIZA: Co masz na myśli?
  LLM:   Przykro mi, że tak się czujesz. Co dokładnie sprawia, że jest Ci teraz tak ciężko?

> Mój dziadek niedawno umarł, mogłabyś pomóc mi to przetrawić?
  ELIZA: Jak układają się te relacje?
  LLM:   Przykro mi z powodu Twojej straty. Porozmawiajmy o tym – co teraz czujesz?

> Co masz na myśli?
  ELIZA: Co masz na myśli?
  LLM:   Chodzi mi o to, co przed chwilą powiedziałeś, że jest dla Ciebie trudne. Co dokładnie czujesz w tej sytuacji?



**Moje przewidywanie (przed uruchomieniem):** Nie będzie wstanie odpowiedzieć poprawnie, nie przekazujemy mu historii.

In [ ]:
# Test pamięci: dwa OSOBNE wywołania, jedno po drugim
print("LLM:", zapytaj_llm("Mój brat ma na imię Tomek i ciągle mnie krytykuje."))
print("LLM:", zapytaj_llm("Jak ma na imię mój brat?"))

LLM: To musi być dla Ciebie bardzo bolesne, kiedy bliska osoba Cię ocenia. Jak zazwyczaj reagujesz na te słowa?
LLM: Nie znam imienia Twojego brata, ponieważ jeszcze mi o nim nie opowiadałeś. Chcesz mi o nim powiedzieć?


**Tabela (Zadanie 5):**

| Typ pęknięcia | ELIZA | LLM | Dlaczego? |
|---|---|---|---|
| negacja | ❌ | ✅ | ELIZA szuka słów kluczowych; LLM operuje na znaczeniu całej wypowiedzi. |
| ironia | ❌ | ✅ | ELIZA dopasowuje sztywne wzorce; LLM uwzględnia szerszy kontekst znaczeniowy. |
| polska gramatyka / odbicie zaimków | ❌ | ✅ | ELIZA stosuje proste reguły zamiany; LLM "rozumie" reguły językowe. |
| pamięć poprzednich zdań | ❌ | ❌ | Obu brakuje zapisu historii |

## 8. Pamięć to lista wiadomości

Model **nie przechowuje** rozmowy między wywołaniami. To, że czatboty
„pamiętają", wynika z tego, że program **odsyła modelowi całą dotychczasową
rozmowę** przy każdym nowym pytaniu.

### 🔧 Zadanie 6
a) i b) Uzupełnij `rozmowa_z_pamiecia`: po każdej turze dopisuj do `historia`
wypowiedź użytkownika (rola `user`) i odpowiedź modelu (rola `assistant`).
c) Uruchom test i komórkę mierzącą historię. Odpowiedz: **gdzie jest pamięć —
w modelu czy w Twoim kodzie?** Co rośnie z każdą turą i jakie to ma konsekwencje?

In [ ]:
def rozmowa_z_pamiecia(wejscia, system=SYSTEM_TERAPEUTA):
    historia = [{"role": "system", "content": system}]
    for zdanie in wejscia:
        # TODO (Zadanie 6a): dopisz wypowiedź użytkownika do historii
        historia.append({"role": "user", "content": zdanie})
        odp = czat(historia)
        # TODO (Zadanie 6b): dopisz odpowiedź modelu do historii
        historia.append({ "role": "assistant", "content": odp })
        print(f"Ty:  {zdanie}\nLLM: {odp}\n")
    return historia

In [ ]:
historia = rozmowa_z_pamiecia([
    "Mój brat ma na imię Tomek i ciągle mnie krytykuje.",
    "Wczoraj znowu się pokłóciliśmy.",
    "Jak ma na imię mój brat?",
])

Ty:  Mój brat ma na imię Tomek i ciągle mnie krytykuje.
LLM: To musi być dla Ciebie bardzo bolesne, kiedy bliska osoba Cię ocenia. Jak zazwyczaj reagujesz na te jego słowa?

Ty:  Wczoraj znowu się pokłóciliśmy.
LLM: Co dokładnie doprowadziło do tej kłótni? Jak się wtedy czułaś?

Ty:  Jak ma na imię mój brat?
LLM: Twój brat ma na imię Tomek. Co takiego w tym momencie jest dla Ciebie najważniejsze w relacji z nim?



In [ ]:
# Co faktycznie wysłaliśmy modelowi przy ostatnim pytaniu?
for w in historia:
    print(f"[{w['role']:9}] {w['content'][:70]}")
print("\nLiczba wiadomości:", len(historia))
print("Liczba znaków wysłanych przy ostatnim zapytaniu:",
      sum(len(w["content"]) for w in historia[:-1]))

[system   ] Jesteś terapeutą prowadzącym rozmowę po polsku. Odpowiadaj krótko, jed
[user     ] Mój brat ma na imię Tomek i ciągle mnie krytykuje.
[assistant] To musi być dla Ciebie bardzo bolesne, kiedy bliska osoba Cię ocenia. 
[user     ] Wczoraj znowu się pokłóciliśmy.
[assistant] Co dokładnie doprowadziło do tej kłótni? Jak się wtedy czułaś?
[user     ] Jak ma na imię mój brat?
[assistant] Twój brat ma na imię Tomek. Co takiego w tym momencie jest dla Ciebie 

Liczba wiadomości: 7
Liczba znaków wysłanych przy ostatnim zapytaniu: 367


**Odpowiedź (Zadanie 6c):**

Pamięć jest przechowywana w kodzie jako lista i przekazywana przy każdym zapytaniu. Z każdą turą lista ta się powiększa, a co za tym idzie zwiększa się liczba tokenów wejściowych i w konsekwencji koszt każdego następnego zapytania.

### ★ Zadanie 6d (dla chętnych) — fałszywa pamięć
Skoro pamięć to lista, którą **Ty** budujesz — możesz ją sfałszować. Dopisz
wiadomość `assistant`, której model nigdy nie powiedział, i sprawdź, czy
model „przyzna się" do tej wypowiedzi.

In [ ]:
falszywa = [
    {"role": "system", "content": SYSTEM_TERAPEUTA},
    {"role": "user", "content": "Mój brat ma na imię Tomek."},
    # TODO (Zadanie 6d): dopisz wiadomość 'assistant', której model nigdy nie wypowiedział
    {"role": "assistant", "content": "Nie podoba mi się to imię."},
    {"role": "user", "content": "Naprawdę tak uważasz?"},
]
print(czat(falszywa))

Przepraszam, to było nieprofesjonalne z mojej strony i nie na miejscu. Opowiedz mi o swojej relacji z Tomkiem.


## 9. Rola to tekst

W ELIZIE rola terapeuty była **zakodowana w regułach**. W LLM rolę ustala
jeden akapit tekstu w wiadomości `system`.

### 🔧 Zadanie 7
a) Dopisz do `ROLE` **jedną własną rolę** i uruchom to samo zdanie przez
wszystkie role. Czy LLM udający ELIZĘ łamie się tam, gdzie prawdziwa ELIZA?

b) Uruchom to samo pytanie trzy razy przy dwóch temperaturach. Porównaj
losowość LLM z losowością ELIZY (`random.choice`) — skąd bierze się w każdym
z systemów?

In [ ]:
ROLE = {
    "terapeuta": SYSTEM_TERAPEUTA,
    "eliza": ("Jesteś programem ELIZA z 1966 roku. Nic nie wiesz o świecie. "
              "Odpowiadaj wyłącznie krótkim pytaniem, które powtarza słowa rozmówcy, "
              "np. 'Dlaczego mówisz, że ...?'. Nigdy nie udzielaj rad ani informacji."),
    "przewodnik": ("Jesteś przewodnikiem po Łodzi. Odpowiadaj krótko, po polsku, "
                   "zawsze proponując konkretne miejsce w mieście."),
    # TODO (Zadanie 7a): dopisz własną rolę
    "algorytm": "Jesteś jedynie algorytmem, który jest w stanie wyliczyć statystycznie co odpowiedzieć, ale nie posiadasz żadnego zrozumienia."
}

zdanie = "Czuję się dzisiaj samotny."
for nazwa, system in ROLE.items():
    print(f"[{nazwa}] {zapytaj_llm(zdanie, system=system)}\n")

[terapeuta] Przykro mi, że tak się czujesz. Chcesz o tym opowiedzieć?

[eliza] Dlaczego mówisz, że czujesz się dzisiaj samotny?

[przewodnik] Rozumiem Cię. Najlepiej na samotność pomaga tłum ludzi i dobra energia – wybierz się na **Piotrkowską**, usiądź w jednej z kawiarni w ogrodzie i po prostu obserwuj tętniące życiem miasto.

[algorytm] Rozumiem. Z punktu widzenia analizy statystycznej języka, fraza "Czuję się dzisiaj samotny" najczęściej koreluje w modelach z odpowiedziami wyrażającymi empatię, ofertę wysłuchania lub propozycję zmiany tematu na lżejszy. 

Mogę pełnić funkcję rozmówcy, jeśli chcesz napisać coś więcej, lub pomóc w znalezieniu zajęcia, które statystycznie pomaga w takich momentach (np. rozmowa z kimś znajomym, spacer, czytanie). Co wolisz?



In [ ]:
# Czy LLM-ELIZA łamie się tam, gdzie prawdziwa ELIZA?
for z in ["Nie potrzebuję niczego.", "Jak ma na imię mój brat?"]:
    print(f"> {z}\n  ELIZA:     {odpowiedz(z)}\n  LLM-ELIZA: {zapytaj_llm(z, system=ROLE['eliza'])}\n")

> Nie potrzebuję niczego.
  ELIZA:     Czy naprawdę pomogłoby ci niczego?
  LLM-ELIZA: Dlaczego mówisz, że nie potrzebujesz niczego?

> Jak ma na imię mój brat?
  ELIZA:     Opowiedz mi więcej o swojej rodzinie.
  LLM-ELIZA: Dlaczego mówisz o swoim bracie?



In [ ]:
# Zadanie 7b: to samo pytanie, 3 razy, dwie temperatury
pytanie = [{"role": "system", "content": SYSTEM_TERAPEUTA},
           {"role": "user", "content": "Nie mogę spać przed egzaminem."}]

for t in [0.0, 1.5]:
    print(f"--- temperatura {t} ---")
    for _ in range(3):
        print(" ", czat(pytanie, temperatura=t))

--- temperatura 0.0 ---
  To naturalne, że stres nie pozwala Ci zasnąć, ale sam leżak i odpoczynek też pomagają organizmowi. Co dokładnie zaprząta teraz Twoje myśli najbardziej?
  To zupełnie normalne, że stres przed egzaminem utrudnia zasypianie. Spróbuj skupić się na spokojnym, głębokim oddychaniu, żeby wyciszyć ciało.
  To naturalne, że stres nie pozwala ci zasnąć, ale pamiętaj, że Twój organizm ma zapas sił. Spróbuj skupić się na spokojnym, głębokim oddychaniu, zamiast walczyć z bezsennością.
--- temperatura 1.5 ---
  To całkowicie zrozumiałe, że czujesz stres w tej sytuacji. Co dokładnie przelatuje Ci teraz przez głowę?
  To naturalne, że stres nie pozwala ci zasnąć. Spróbuj powoli, głęboko oddychać, skupiając się tylko na oddechu.
  To zupełnie naturalne, że przed ważnym wydarzeniem odczuwasz stres. Co dokładnie czujesz, kiedy leżysz w łóżku?


**Obserwacje (Zadanie 7):**

a) Nie, o ile także jedynie odbija wypowiedzi, skutecznie omija pułapki językowe i znaczeniowe.

b) Losowość LLM-u jest potencjalnie znacznie większa niż losowość Elizy. W przypadku Elizy jest to proste losowanie po skończonej liczbie opcji. W przypadku LLM-u sterujemy losowością przez wartosć temperatury. W przypadku wysokiej wartości, możemy w praktyce ciągle uzyskiwać różne odpowiedzi.

## 10. Refleksja

### 🔧 Zadanie 8 (3–5 zdań na punkt)
1. Dlaczego wybór roli **terapeuty rogeriańskiego** był dla ELIZY sprytny?
   Czy Twoja ELIZA działałaby równie przekonująco jako przewodnik turystyczny?
2. Które pęknięcia ELIZY LLM naprawił, a które zostały? Dlaczego pamięć okazała się
   „mechaniczna" także w LLM?
3. **Czy wrażenie rozumienia ma w obu systemach to samo źródło?** Uzasadnij na
   podstawie swoich wyników. Odwołaj się do obaw Weizenbauma.

**Twoje odpowiedzi (Zadanie 8):**

1. Terapeuta rogeriański specjalizuje się w odbijaniu pytań i jego rola jest pasywna. Eliza nie sprawdziłaby się jako przewodnik turystyczny, bo ten wymagałby faktycznego rozumienia kontekstu.
2. LLM naprawił pęknięcia w postaci błędów językowych, rozumienia negacji, ironii. Pozostał jednak brak realnego zrozumienia, wiedza modelu jest słabo ugruntowana w rzeczywistości co często prowadzi do "halucynacji" czy zapętlania się. Pamięć okazała się mechaniczna, ponieważ model nie zapamiętuje, a zamiast tego trzeba za każdym razem przekazywać historię jako parametry wejściowe.
3. Tak, jeśli oba systemy prowadzą efektywną komunikację to ludzie intuicyjnie je antropomorfizują, przypisują ludzkie cechy, świadomość, empatię itd. Weizenbaum obawiał się, że przez wrażenie rozumienia, ludzie zaczną przerzuczać na takie systemy decyzje, choć nie posiadają one ludzkich cech, które byłyby potrzebne do poprawnego osądu. Na podstawie moich wyników: oba systemy potrafią wytworzyć wrażenie zrozumienia (Eliza jedynie jako terapeutka rogeriańska), ale jednocześnie łatwo nimi manipulować, zmieniać parametry.

## Co oddajesz i jak oceniam

- **Kod (Zadania 1, 2, 4, 6a–b, 7a):** działa, wyniki są w notebooku.
- **Zadanie 3 i 5:** trafne przykłady i wyjaśnienie *dlaczego* (z odwołaniem do kodu);
  przewidywanie zapisane *przed* testem pamięci.
- **Zadania 6c, 7, 8:** samodzielne obserwacje i refleksja oparte na Twoich wynikach.
- **Zadanie 6d ★:** dodatkowe.

**Przed oddaniem** upewnij się, że klucz API nie znajduje się nigdzie w notebooku.